# Factor Momentum: Empirical Analysis

This notebook reproduces the empirical analysis in the thesis. The structure
follows the thesis:

1. Data preparation (Section 3) — load JKP factors, build regional panels.
2. Monte Carlo simulation on theme aggregation (Section 4.1 / 5.1).
3. Autocorrelation diagnostics on single factors (Section 4.2 / 5.2.1).
4. AR(1) estimation on single factors (Section 4.3 / 5.2.2).
5. One-month factor momentum strategies (Section 4.5 / 5.3).
6. Performance evaluation and alpha vs the equal-weight benchmark (Section 4.6 / 5.3.2).
7. Long-leg / short-leg decomposition (Section 4.6.3 / 5.3.3).
8. TSMOM vs CSMOM spanning regressions (Section 4.6.4).
9. Lo–MacKinlay / Lewellen decomposition of momentum profits (Section 4.7 / 5.4).
10. Rolling and conditional-state analysis (Section 4.8.1 / 5.5.1).
11. Decade subsample stability (Section 4.8.2 / 5.5.2).
12. Transaction costs and net performance (Section 4.8.3 / 5.5.3).

## Imports

In [ ]:
import warnings
from collections import OrderedDict
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.backends.backend_pdf import PdfPages

import statsmodels.api as sm
from statsmodels.tsa.stattools import acf
from scipy.stats import binomtest, skew as _skew

warnings.filterwarnings("ignore")

## 1. Data

The JKP factor library is loaded and four regional panels are constructed
(USA, EMEA, APAC, Nordics) following Section 3.2.2 of the thesis. Non-U.S.
regional factor returns are built as stock-count–weighted averages of the
country-level series.

In [ ]:
dataset = pd.read_csv("jkp_factors.csv")
dataset["date"] = pd.to_datetime(dataset["date"], errors="coerce")

In [ ]:
nordics = ["dnk", "fin", "nor", "swe"]
emea = [
    "aut", "bel", "che", "deu", "dnk", "esp", "fin", "fra", "gbr",
    "irl", "isr", "ita", "nld", "nor", "prt", "swe",
]
apac = ["aus", "hkg", "jpn", "nzl", "sgp", "kor", "twn"]

locs_present = set(dataset["location"].dropna().unique())
nordics = [c for c in nordics if c in locs_present]
emea = [c for c in emea if c in locs_present]
apac = [c for c in apac if c in locs_present]

In [ ]:
factor_order = dataset["name"].drop_duplicates().tolist()


def pivot_country_metric(df, value_col, factor_cols):
    wide = (
        df.pivot(index=["location", "date"], columns="name", values=value_col)
        .reindex(columns=factor_cols)
        .sort_index()
        .reset_index()
    )
    wide.columns.name = None
    return wide


def agg_region_from_wide(ret_wide, weight_wide, loc_list, region_name, factor_cols):
    key_cols = ["location", "date"]
    ret_sub = ret_wide.loc[ret_wide["location"].isin(loc_list), key_cols + factor_cols].copy()
    w_sub = weight_wide.loc[weight_wide["location"].isin(loc_list), key_cols + factor_cols].copy()
    merged = ret_sub.merge(w_sub, on=key_cols, how="inner", suffixes=("_ret", "_w"))

    ret_cols = [f"{f}_ret" for f in factor_cols]
    w_cols = [f"{f}_w" for f in factor_cols]

    rows = []
    for dt, g in merged.groupby("date", sort=True):
        ret_mat = g[ret_cols].to_numpy(dtype=float)
        w_mat = g[w_cols].to_numpy(dtype=float)
        mask = np.isfinite(ret_mat) & np.isfinite(w_mat) & (w_mat != 0)
        num = np.where(mask, ret_mat * w_mat, 0.0).sum(axis=0)
        den = np.where(mask, w_mat, 0.0).sum(axis=0)
        vals = np.divide(num, den, out=np.full(len(factor_cols), np.nan), where=den != 0)
        rows.append([region_name, dt, *vals])

    return pd.DataFrame(rows, columns=["location", "date", *factor_cols])

In [ ]:
dataset_wide = pivot_country_metric(dataset, "ret", factor_order)
n_stocks_wide = pivot_country_metric(dataset, "n_stocks", factor_order)

# USA panel kept from raw country data, starting in 1986 to match the regional samples.
usa_factors = (
    dataset_wide.loc[dataset_wide["location"].eq("usa")]
    .assign(date=lambda x: pd.to_datetime(x["date"], errors="coerce"))
    .loc[lambda x: x["date"] >= "1986-01-01"]
    .sort_values("date")
    .reset_index(drop=True)
)

# Non-U.S. regions: stock-count-weighted averages of country-level factor returns.
country_factors = (
    dataset_wide
    .assign(date=lambda x: pd.to_datetime(x["date"], errors="coerce"))
    .loc[lambda x: x["date"] >= "1986-01-01"]
    .sort_values(["location", "date"])
    .reset_index(drop=True)
)
country_factors_nstocks = (
    n_stocks_wide
    .assign(date=lambda x: pd.to_datetime(x["date"], errors="coerce"))
    .loc[lambda x: x["date"] >= "1986-01-01"]
    .sort_values(["location", "date"])
    .reset_index(drop=True)
)

factor_cols = [c for c in country_factors.columns if c not in ["location", "date"]]

nordics_factors = agg_region_from_wide(country_factors, country_factors_nstocks, nordics, "nordics", factor_cols)
emea_factors = agg_region_from_wide(country_factors, country_factors_nstocks, emea, "emea", factor_cols)
apac_factors = agg_region_from_wide(country_factors, country_factors_nstocks, apac, "apac", factor_cols)

region_series_map = {
    "usa": usa_factors,
    "emea": emea_factors,
    "apac": apac_factors,
    "nordics": nordics_factors,
}

for r, df in region_series_map.items():
    print(f"{r:<8s} shape: {df.shape}")

## 2. Monte Carlo simulation on theme aggregation

Section 4.1 motivates working at the level of individual factors rather than at
the level of JKP themes. Each simulated factor return is the sum of a
factor-specific AR(1) component and a common transitory disturbance; the theme
return is the equal-weighted average of $K$ such factor series. The parameter
$\rho$ controls how aligned the persistent innovations are across factors
inside the theme.

In [ ]:
SIM_CFG = {
    "n_months": 480,
    "burn_in": 100,
    "n_factors": 5,
    "phi": 0.30,
    "sigma_u": 0.05,
    "sigma_eta": 0.06,
    "rho_grid": np.linspace(0.0, 0.95, 11),
    "n_sim": 500,
    "seed": 42,
}


def lag1_autocorr(x):
    x = np.asarray(x, dtype=float)
    x0 = x[:-1] - x[:-1].mean()
    x1 = x[1:] - x[1:].mean()
    denom = np.sqrt((x0 @ x0) * (x1 @ x1))
    return np.nan if denom == 0 else (x0 @ x1) / denom


def mean_pairwise_corr(mat):
    c = np.corrcoef(mat.T)
    iu = np.triu_indices_from(c, k=1)
    return np.nanmean(c[iu])


def simulate_theme(phi, sigma_u, sigma_eta, rho, n_factors, n_months, burn_in, rng):
    T = n_months + burn_in
    cov = np.full((n_factors, n_factors), rho * sigma_u**2)
    np.fill_diagonal(cov, sigma_u**2)

    x = np.zeros((T, n_factors))
    for t in range(1, T):
        u_t = rng.multivariate_normal(mean=np.zeros(n_factors), cov=cov)
        x[t] = phi * x[t - 1] + u_t

    eta = sigma_eta * rng.standard_normal(T)
    r = (x + eta[:, None])[burn_in:]
    return r, r.mean(axis=1)

In [ ]:
rng_master = np.random.default_rng(SIM_CFG["seed"])
rows = []

for rho in SIM_CFG["rho_grid"]:
    individual_acfs, theme_acfs, within_corrs = [], [], []

    for _ in range(SIM_CFG["n_sim"]):
        rng = np.random.default_rng(rng_master.integers(0, 10_000_000))
        factor_ret, theme_ret = simulate_theme(
            phi=SIM_CFG["phi"],
            sigma_u=SIM_CFG["sigma_u"],
            sigma_eta=SIM_CFG["sigma_eta"],
            rho=rho,
            n_factors=SIM_CFG["n_factors"],
            n_months=SIM_CFG["n_months"],
            burn_in=SIM_CFG["burn_in"],
            rng=rng,
        )
        individual_acfs.append(np.mean([lag1_autocorr(factor_ret[:, j]) for j in range(factor_ret.shape[1])]))
        theme_acfs.append(lag1_autocorr(theme_ret))
        within_corrs.append(mean_pairwise_corr(factor_ret))

    rows.append({
        "rho": rho,
        "avg_within_theme_corr": np.mean(within_corrs),
        "avg_individual_lag1_acf": np.mean(individual_acfs),
        "theme_lag1_acf": np.mean(theme_acfs),
    })

sim_df = pd.DataFrame(rows)
display(sim_df.round(3))

In [ ]:
navy = "#0B1F3A"
anthracite = "#B8B8B8"

plt.figure(figsize=(8.5, 5.2))
plt.plot(
    sim_df["avg_within_theme_corr"], sim_df["avg_individual_lag1_acf"],
    marker="o", markersize=6, linewidth=2.4, color=navy,
    markeredgecolor="white", markeredgewidth=0.8,
    label="Average individual factor lag-1 ACF",
)
plt.plot(
    sim_df["avg_within_theme_corr"], sim_df["theme_lag1_acf"],
    marker="^", markersize=7, linewidth=2.4, color=anthracite,
    markeredgecolor="white", markeredgewidth=0.8,
    label="Grouped theme lag-1 ACF",
)
plt.xlabel("Average within-theme factor correlation")
plt.ylabel("Lag-1 autocorrelation")
plt.title("Grouping weakens persistence when factors are insufficiently aligned")
plt.grid(True, alpha=0.25)
plt.legend(frameon=False)
plt.tight_layout()
plt.show()

## 3. Autocorrelation diagnostics on single factors

For each region and each of the 153 factors we compute the sample
autocorrelation function up to lag 12, with the effective maximum lag adjusted
for short samples. The lag-1 coefficient $\hat\rho_{i,r}(1)$ and its
large-sample $t$-statistic $\hat\rho_{i,r}(1)\sqrt{T}$ are the primary
diagnostics. Within-factor averages over lags 1–3, 1–6 and 1–12 are stored
for the appendix.

In [ ]:
def compute_autocorr_stats(series, max_lag=12):
    s = pd.Series(series).dropna().astype(float).reset_index(drop=True)
    T = len(s)
    if T < 10:
        return None

    max_lag_eff = min(max_lag, T // 2 - 1, T - 2)
    if max_lag_eff < 1:
        return None

    lags = np.arange(1, max_lag_eff + 1)
    acf_vals = acf(s, nlags=max_lag_eff, fft=False)[1:]
    t_stats = acf_vals * np.sqrt(T)

    return {
        "T": T,
        "lags": lags,
        "acf": acf_vals,
        "t_stats": t_stats,
        "crit_t": 1.96,
        "crit_corr": 1.96 / np.sqrt(T),
    }


summary_rows = []
for region_name, df_region in region_series_map.items():
    df_region = df_region.sort_values("date").copy()
    for factor_col in factor_cols:
        stats = compute_autocorr_stats(df_region[factor_col], max_lag=12)
        if stats is None:
            summary_rows.append({
                "region": region_name, "factor": factor_col,
                "T": np.nan, "lag1_acf": np.nan, "lag1_tstat": np.nan,
                "lag1_sig_pos": False, "lag1_sig_neg": False,
                "mean_acf_1_3": np.nan, "mean_acf_1_6": np.nan, "mean_acf_1_12": np.nan,
                "mean_acf_2_6": np.nan, "mean_acf_2_12": np.nan,
            })
            continue

        acf_vals, t_vals, T = stats["acf"], stats["t_stats"], stats["T"]
        summary_rows.append({
            "region": region_name, "factor": factor_col, "T": T,
            "lag1_acf": acf_vals[0],
            "lag1_tstat": t_vals[0],
            "lag1_sig_pos": bool(t_vals[0] > 1.96),
            "lag1_sig_neg": bool(t_vals[0] < -1.96),
            "mean_acf_1_3": np.nanmean(acf_vals[:3]),
            "mean_acf_1_6": np.nanmean(acf_vals[:6]) if len(acf_vals) >= 1 else np.nan,
            "mean_acf_1_12": np.nanmean(acf_vals[:12]) if len(acf_vals) >= 1 else np.nan,
            "mean_acf_2_6": np.nanmean(acf_vals[1:6]) if len(acf_vals) >= 6 else np.nan,
            "mean_acf_2_12": np.nanmean(acf_vals[1:12]) if len(acf_vals) >= 12 else np.nan,
        })

autocorr_summary = pd.DataFrame(summary_rows)

### Region-level aggregate persistence

For each region we report the cross-sectional mean and median of
$\hat\rho_{i,r}(1)$, the share of factors with a positive coefficient, and the
exact two-sided binomial $p$-value testing whether that share differs from
$0.5$ under the null of no autocorrelation.

In [ ]:
agg_rows = []
for region_name in region_series_map.keys():
    sub = autocorr_summary.loc[autocorr_summary["region"] == region_name].dropna(subset=["lag1_acf"])
    n = len(sub)
    n_pos = int((sub["lag1_acf"] > 0).sum())
    binom_p = binomtest(n_pos, n, p=0.5, alternative="two-sided").pvalue if n > 0 else np.nan

    agg_rows.append({
        "region": region_name,
        "n_factors": n,
        "mean_lag1_acf": sub["lag1_acf"].mean(),
        "median_lag1_acf": sub["lag1_acf"].median(),
        "std_lag1_acf": sub["lag1_acf"].std(),
        "mean_lag1_tstat": sub["lag1_tstat"].mean(),
        "mean_acf_1_3": sub["mean_acf_1_3"].mean(),
        "mean_acf_1_6": sub["mean_acf_1_6"].mean(),
        "mean_acf_1_12": sub["mean_acf_1_12"].mean(),
        "share_pos": n_pos / n if n else np.nan,
        "binom_p_share_pos_vs_half": binom_p,
    })

region_aggregate = pd.DataFrame(agg_rows).set_index("region")
display(region_aggregate.round(4))

### Skip-month decomposition

For each region, the share of the 1–6 and 1–12 window averages attributable to
lag 1 alone. Reported as Appendix Table A1 in the thesis to motivate the
one-month signal horizon (Section 4.4).

In [ ]:
decomp_rows = []
for region_name in region_series_map.keys():
    sub = autocorr_summary.loc[autocorr_summary["region"] == region_name].dropna(subset=["lag1_acf"])

    mean_lag1 = sub["lag1_acf"].mean()
    mean_1_6 = sub["mean_acf_1_6"].mean()
    mean_2_6 = sub["mean_acf_2_6"].mean()
    mean_1_12 = sub["mean_acf_1_12"].mean()
    mean_2_12 = sub["mean_acf_2_12"].mean()

    share_lag1_in_1_6 = (mean_lag1 / 6) / mean_1_6 * 100 if pd.notna(mean_1_6) and mean_1_6 != 0 else np.nan
    share_lag1_in_1_12 = (mean_lag1 / 12) / mean_1_12 * 100 if pd.notna(mean_1_12) and mean_1_12 != 0 else np.nan

    decomp_rows.append({
        "region": region_name,
        "n_factors": len(sub),
        "mean_lag1_acf": mean_lag1,
        "mean_acf_1_6": mean_1_6,
        "mean_acf_2_6": mean_2_6,
        "mean_acf_1_12": mean_1_12,
        "mean_acf_2_12": mean_2_12,
        "share_lag1_in_1_6_pct": share_lag1_in_1_6,
        "share_lag1_in_1_12_pct": share_lag1_in_1_12,
    })

region_decomposition = pd.DataFrame(decomp_rows).set_index("region")
display(region_decomposition.round(4))

### Cross-regional Spearman rank correlation of lag-1 ACF

Tests whether the ordering of factor persistence is consistent across regions
(Section 4.2.5).

In [ ]:
lag1_pivot = (
    autocorr_summary
    .pivot(index="factor", columns="region", values="lag1_acf")
    .reindex(columns=["usa", "emea", "apac", "nordics"])
)

print("Spearman rank correlation of lag-1 ACF across regions:")
display(lag1_pivot.corr(method="spearman").round(2))

### Appendix tables and per-factor ACF plots

Full 12-lag ACF tables (one per region) are exported for the appendix, and
PDFs of the top and bottom 15 factors by lag-1 ACF are saved for each region.

In [ ]:
output_dir = Path("autocorr_factor_plots")
output_dir.mkdir(exist_ok=True)


def plot_autocorr_two_panel(df_wide, region_name, factor_col, max_lag=12):
    df_plot = df_wide.sort_values("date").copy()
    stats = compute_autocorr_stats(df_plot[factor_col], max_lag=max_lag)
    if stats is None:
        return None

    lags = stats["lags"]
    fig, axes = plt.subplots(1, 2, figsize=(12, 4.6))
    fig.suptitle(f"{region_name.upper()} — {factor_col}")

    axes[0].bar(lags, stats["t_stats"], width=0.8, edgecolor="black", linewidth=0.4)
    axes[0].axhline(0, linewidth=0.8)
    axes[0].axhline(stats["crit_t"], linestyle=":", linewidth=1.0)
    axes[0].axhline(-stats["crit_t"], linestyle=":", linewidth=1.0)
    axes[0].set_title(f"t-stat by lag (T = {stats['T']})")
    axes[0].set_xlabel("Month lag"); axes[0].set_ylabel("t-statistic")

    axes[1].bar(lags, stats["acf"], width=0.8, edgecolor="black", linewidth=0.4)
    axes[1].axhline(0, linewidth=0.8)
    axes[1].axhline(stats["crit_corr"], linestyle=":", linewidth=1.0)
    axes[1].axhline(-stats["crit_corr"], linestyle=":", linewidth=1.0)
    axes[1].set_title("ACF")
    axes[1].set_xlabel("Month lag"); axes[1].set_ylabel("Autocorrelation")

    plt.tight_layout()
    return fig


for region_name, df_region in region_series_map.items():
    sub = autocorr_summary.loc[autocorr_summary["region"] == region_name].dropna(subset=["lag1_acf"]).sort_values("lag1_acf")
    selected = sub.head(15)["factor"].tolist() + sub.tail(15)["factor"].tolist()

    pdf_path = output_dir / f"{region_name}_factors_autocorr.pdf"
    with PdfPages(pdf_path) as pdf:
        for factor_col in selected:
            fig = plot_autocorr_two_panel(df_region, region_name, factor_col, max_lag=12)
            if fig is not None:
                pdf.savefig(fig, bbox_inches="tight")
                plt.close(fig)
    print(f"Saved: {pdf_path}")

In [ ]:
def build_full_lag_table(df_region, factor_cols, max_lag=12):
    rows = []
    df_region = df_region.sort_values("date").copy()
    for f in factor_cols:
        stats = compute_autocorr_stats(df_region[f], max_lag=max_lag)
        if stats is None:
            rows.append({"factor": f, "T": np.nan, **{f"rho_{k}": np.nan for k in range(1, max_lag + 1)}})
            continue
        row = {"factor": f, "T": stats["T"]}
        for k in range(1, max_lag + 1):
            if k - 1 < len(stats["acf"]):
                rho = stats["acf"][k - 1]
                t = stats["t_stats"][k - 1]
                star = "***" if abs(t) > 2.58 else ("**" if abs(t) > 1.96 else ("*" if abs(t) > 1.645 else ""))
                row[f"rho_{k}"] = f"{rho:+.3f}{star}"
            else:
                row[f"rho_{k}"] = ""
        rows.append(row)
    return pd.DataFrame(rows).set_index("factor")


for region_name, df_region in region_series_map.items():
    tbl = build_full_lag_table(df_region, factor_cols, max_lag=12)
    out_path = output_dir / f"appendix_full_acf_{region_name}.csv"
    tbl.to_csv(out_path)
    print(f"Saved appendix ACF table: {out_path}  (shape={tbl.shape})")

## 4. AR(1) estimation on single factors

For each factor in each region we estimate

$$f_{i,r,t} = \alpha_{i,r} + \phi_{i,r} f_{i,r,t-1} + \varepsilon_{i,r,t}$$

by OLS with Newey–West HAC standard errors at one lag, requiring at least 30
monthly observations per factor.

In [ ]:
def pvalue_to_stars(p):
    if pd.isna(p):
        return ""
    if p < 0.001: return "***"
    if p < 0.01:  return "**"
    if p < 0.05:  return "*"
    if p < 0.1:   return "."
    return ""


def format_coef_with_stars(coef, pval, digits=4):
    if pd.isna(coef):
        return ""
    return f"{coef:.{digits}f}{pvalue_to_stars(pval)}"


def fit_ar1(series):
    s = pd.Series(series).dropna().astype(float).reset_index(drop=True)
    if len(s) < 30:
        return {"n_obs": len(s), "model": None}

    y = s.iloc[1:].reset_index(drop=True)
    x = s.shift(1).iloc[1:].reset_index(drop=True)
    X = sm.add_constant(x); X.columns = ["const", "lag1"]

    try:
        model = sm.OLS(y, X).fit(cov_type="HAC", cov_kwds={"maxlags": 1})
        return {"n_obs": len(y), "model": model}
    except Exception:
        return {"n_obs": len(y), "model": None}


def run_ar1_panel(df_wide, factor_cols, region_name):
    rows = []
    df_wide = df_wide.sort_values("date").copy()
    for factor in factor_cols:
        res = fit_ar1(df_wide[factor])
        m = res["model"]
        if m is None:
            rows.append({
                "region": region_name, "factor": factor, "n_obs": res["n_obs"],
                "ar1_raw": np.nan, "ar1_t": np.nan, "ar1_p": np.nan, "r2": np.nan,
            })
            continue
        rows.append({
            "region": region_name, "factor": factor, "n_obs": res["n_obs"],
            "ar1_raw": m.params.get("lag1", np.nan),
            "ar1_t": m.tvalues.get("lag1", np.nan),
            "ar1_p": m.pvalues.get("lag1", np.nan),
            "r2": m.rsquared,
        })
    return pd.DataFrame(rows)


ar1_results_by_region = {}
all_ar1 = []
for region_name, df_region in region_series_map.items():
    res = run_ar1_panel(df_region, factor_cols, region_name)
    ar1_results_by_region[region_name] = res
    all_ar1.append(res)

ar1_results_all = pd.concat(all_ar1, axis=0, ignore_index=True)

### Regional AR(1) summary (Table 5.4)

In [ ]:
region_order = ["usa", "emea", "apac", "nordics"]

summary_rows = []
for region_name in region_order:
    sub = ar1_results_by_region[region_name].dropna(subset=["ar1_raw"])
    n = len(sub)
    n_pos = int((sub["ar1_raw"] > 0).sum())
    n_sig_pos_5 = int(((sub["ar1_raw"] > 0) & (sub["ar1_p"] < 0.05)).sum())
    n_sig_neg_5 = int(((sub["ar1_raw"] < 0) & (sub["ar1_p"] < 0.05)).sum())

    summary_rows.append({
        "region": region_name,
        "N_factors": n,
        "mean_ar1": sub["ar1_raw"].mean(),
        "median_ar1": sub["ar1_raw"].median(),
        "n_ar1_positive": n_pos,
        "share_ar1_positive": n_pos / n if n else np.nan,
        "n_sig_pos_5pct": n_sig_pos_5,
        "n_sig_neg_5pct": n_sig_neg_5,
    })

region_ar1_summary = pd.DataFrame(summary_rows)
display(region_ar1_summary.round(4))

### Distribution of AR(1) coefficients by region (Figure 5.2)

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(12, 8), sharex=True, sharey=True)
axes = axes.flatten()

for ax, region_name in zip(axes, region_order):
    sub = ar1_results_all.loc[ar1_results_all["region"] == region_name, "ar1_raw"].dropna()
    ax.hist(sub, bins=20, edgecolor="black", linewidth=0.6, alpha=0.8)
    ax.axvline(0, linestyle="--", linewidth=1.0)
    ax.axvline(sub.mean(), linestyle="-", linewidth=1.2, label=f"mean = {sub.mean():.3f}")
    ax.axvline(sub.median(), linestyle=":", linewidth=1.2, label=f"median = {sub.median():.3f}")
    ax.set_title(region_name.upper())
    ax.set_xlabel("AR(1) coefficient")
    ax.set_ylabel("Number of factors")
    ax.legend(fontsize=8)

fig.suptitle("Distribution of factor AR(1) coefficients by region", y=1.02)
plt.tight_layout()
plt.show()

## 5. One-month factor momentum strategies

Four active long–short strategies are built (Section 4.5), plus the
equal-weight benchmark:

- **TSMOM sign** — signal $\operatorname{sign}(r_{i,t-1})$
- **TSMOM scaled** — signal $r_{i,t-1}$
- **CSMOM discrete** — median split on $r_{i,t-1}$ (top half long, bottom half short)
- **CSMOM continuous** — signal $r_{i,t-1} - \bar r_{t-1}$

All four use the common gross-exposure normalization
$w_{i,t} = s_{i,t-1} / \sum_j |s_{j,t-1}|$.

In [ ]:
def get_factor_cols(panel):
    return [c for c in panel.columns if c not in ["location", "date"]]


def normalize_weights(weights):
    gross = weights.abs().sum(axis=1).replace(0, np.nan)
    return weights.div(gross, axis=0)


def equal_weight_factor_benchmark(panel):
    fcols = get_factor_cols(panel)
    out = panel[["date"]].copy()
    out["ew_factor_ret"] = panel[fcols].mean(axis=1, skipna=True)
    return out


def get_lagged_factor_returns(panel, lag=1):
    fcols = get_factor_cols(panel)
    df = panel.sort_values("date").copy()
    lagged = df[fcols].shift(lag)
    lagged.index = df.index
    return df, fcols, lagged


def build_tsmom_sign_1m(panel):
    df, fcols, lagged = get_lagged_factor_returns(panel, lag=1)
    signal = np.sign(lagged).where(lagged.notna(), np.nan)
    weights = normalize_weights(signal)
    out = df[["date"]].copy()
    out["tsmom_sign_1m"] = (weights * df[fcols]).sum(axis=1, min_count=1)
    return out, weights


def build_tsmom_scaled_1m(panel):
    df, fcols, lagged = get_lagged_factor_returns(panel, lag=1)
    weights = normalize_weights(lagged)
    out = df[["date"]].copy()
    out["tsmom_scaled_1m"] = (weights * df[fcols]).sum(axis=1, min_count=1)
    return out, weights


def _median_split_weights_row(row):
    s = row.dropna().sort_values()
    w = pd.Series(0.0, index=row.index)
    n = len(s)
    if n < 2:
        w[:] = np.nan
        return w
    k = n // 2
    w.loc[s.index[:k]] = -1.0
    w.loc[s.index[-k:]] = 1.0
    return w


def build_csmom_discrete_1m(panel):
    df, fcols, lagged = get_lagged_factor_returns(panel, lag=1)
    signal = lagged.apply(_median_split_weights_row, axis=1)
    weights = normalize_weights(signal)
    out = df[["date"]].copy()
    out["csmom_discrete_1m"] = (weights * df[fcols]).sum(axis=1, min_count=1)
    return out, weights


def build_csmom_continuous_1m(panel):
    df, fcols, lagged = get_lagged_factor_returns(panel, lag=1)
    row_means = lagged.mean(axis=1, skipna=True)
    signal = lagged.sub(row_means, axis=0)
    weights = normalize_weights(signal)
    out = df[["date"]].copy()
    out["csmom_continuous_1m"] = (weights * df[fcols]).sum(axis=1, min_count=1)
    return out, weights


def build_factor_momentum_panel(panel):
    df = panel.sort_values("date").copy()
    ew = equal_weight_factor_benchmark(df)
    ts_sign, w_ts_sign = build_tsmom_sign_1m(df)
    ts_scaled, w_ts_scaled = build_tsmom_scaled_1m(df)
    cs_disc, w_cs_disc = build_csmom_discrete_1m(df)
    cs_cont, w_cs_cont = build_csmom_continuous_1m(df)

    out = ew.merge(ts_sign, on="date", how="left")
    out = out.merge(ts_scaled, on="date", how="left")
    out = out.merge(cs_disc, on="date", how="left")
    out = out.merge(cs_cont, on="date", how="left")

    weights = {
        "tsmom_sign_1m": w_ts_sign,
        "tsmom_scaled_1m": w_ts_scaled,
        "csmom_discrete_1m": w_cs_disc,
        "csmom_continuous_1m": w_cs_cont,
    }
    return out, weights

## 6. Performance evaluation and alpha vs the EW benchmark

For each strategy we report annualized mean and volatility, Sharpe and Sortino
ratios, the $t$-statistic of the monthly mean, hit rate, P&L ratio, and
cumulative return. The alpha against the equal-weight benchmark is estimated
from the HAC regression

$$r_t^{\text{strat}} = \alpha + \beta\, r_t^{\text{EW}} + \varepsilon_t.$$

In [ ]:
def performance_summary(ret, periods_per_year=12, target_return=0.0):
    s = pd.Series(ret).dropna().astype(float)
    if len(s) == 0:
        return {k: np.nan for k in [
            "n_months", "ann_mean", "ann_vol", "ann_sharpe", "ann_sortino",
            "mean_tstat", "hit_rate", "pnl_ratio", "cum_return",
        ]}

    mean_m = s.mean()
    vol_m = s.std(ddof=1)
    downside = s[s < target_return] - target_return
    downside_dev_m = np.sqrt((downside**2).mean()) if len(downside) > 0 else 0.0

    ann_mean = mean_m * periods_per_year
    ann_vol = vol_m * np.sqrt(periods_per_year)
    ann_sharpe = ann_mean / ann_vol if ann_vol > 0 else np.nan
    ann_sortino = ann_mean / (downside_dev_m * np.sqrt(periods_per_year)) if downside_dev_m > 0 else np.nan
    mean_tstat = mean_m / (vol_m / np.sqrt(len(s))) if vol_m > 0 else np.nan
    hit_rate = (s > 0).mean()
    pos, neg = s[s > 0], s[s < 0]
    pnl_ratio = (pos.mean() / abs(neg.mean())) if len(neg) > 0 and neg.mean() < 0 else np.nan

    return {
        "n_months": int(len(s)),
        "ann_mean": float(ann_mean),
        "ann_vol": float(ann_vol),
        "ann_sharpe": float(ann_sharpe) if np.isfinite(ann_sharpe) else np.nan,
        "ann_sortino": float(ann_sortino) if np.isfinite(ann_sortino) else np.nan,
        "mean_tstat": float(mean_tstat) if np.isfinite(mean_tstat) else np.nan,
        "hit_rate": float(hit_rate),
        "pnl_ratio": float(pnl_ratio) if np.isfinite(pnl_ratio) else np.nan,
        "cum_return": float((1.0 + s).prod() - 1.0),
    }


def hac_alpha(y, x=None, maxlags=6):
    df = pd.DataFrame({"y": y})
    if x is not None:
        X = x.copy(); X.columns = [f"x{i}" for i in range(X.shape[1])]
        df = pd.concat([df, X], axis=1)
    df = df.dropna()
    if len(df) == 0:
        return {"alpha": np.nan, "alpha_tstat": np.nan, "beta": np.nan, "beta_tstat": np.nan}

    X = sm.add_constant(df.drop(columns="y"), has_constant="add")
    fit = sm.OLS(df["y"], X).fit(cov_type="HAC", cov_kwds={"maxlags": maxlags})
    return {
        "alpha": float(fit.params["const"]),
        "alpha_tstat": float(fit.tvalues["const"]),
        "beta": float(fit.params["x0"]) if "x0" in fit.params else np.nan,
        "beta_tstat": float(fit.tvalues["x0"]) if "x0" in fit.tvalues else np.nan,
    }


def summarize_factor_momentum_panel(panel, region_name):
    strategy_cols = [
        "ew_factor_ret", "tsmom_sign_1m", "tsmom_scaled_1m",
        "csmom_discrete_1m", "csmom_continuous_1m",
    ]
    rows = []
    for strat in strategy_cols:
        perf = performance_summary(panel[strat])
        if strat == "ew_factor_ret":
            alpha_res = {"alpha": np.nan, "alpha_tstat": np.nan, "beta": np.nan, "beta_tstat": np.nan}
            corr_vs_ew = 1.0
        else:
            alpha_res = hac_alpha(panel[strat], panel[["ew_factor_ret"]], maxlags=6)
            corr_vs_ew = panel[[strat, "ew_factor_ret"]].corr().iloc[0, 1]

        alpha_m = alpha_res["alpha"]
        rows.append({
            "region": region_name, "strategy": strat, **perf,
            "alpha_m_vs_ew": alpha_m,
            "alpha_ann_vs_ew": 12 * alpha_m if pd.notna(alpha_m) else np.nan,
            "alpha_t_vs_ew": alpha_res["alpha_tstat"],
            "beta_vs_ew": alpha_res.get("beta", np.nan),
            "beta_t_vs_ew": alpha_res.get("beta_tstat", np.nan),
            "corr_vs_ew": corr_vs_ew,
        })
    return pd.DataFrame(rows)

In [ ]:
strategy_order = [
    "ew_factor_ret",
    "tsmom_sign_1m", "tsmom_scaled_1m",
    "csmom_discrete_1m", "csmom_continuous_1m",
]

strategy_labels = {
    "ew_factor_ret": "EW benchmark",
    "tsmom_sign_1m": "TSMOM sign (1-1)",
    "tsmom_scaled_1m": "TSMOM scaled (1-1)",
    "csmom_discrete_1m": "CSMOM discrete (1-1)",
    "csmom_continuous_1m": "CSMOM continuous (1-1)",
}

momentum_panels_by_region = {}
momentum_weights_by_region = {}
momentum_summary_list = []

for region_name, df_region in region_series_map.items():
    strat_panel, strat_weights = build_factor_momentum_panel(df_region)
    summary_df = summarize_factor_momentum_panel(strat_panel, region_name)

    momentum_panels_by_region[region_name] = strat_panel
    momentum_weights_by_region[region_name] = strat_weights
    momentum_summary_list.append(summary_df)

momentum_summary_all = pd.concat(momentum_summary_list, axis=0, ignore_index=True)
momentum_summary_all["strategy"] = pd.Categorical(
    momentum_summary_all["strategy"], categories=strategy_order, ordered=True,
)
momentum_summary_all = momentum_summary_all.sort_values(["region", "strategy"]).reset_index(drop=True)
momentum_summary_all["strategy_label"] = momentum_summary_all["strategy"].map(strategy_labels)

In [ ]:
summary_cols = [
    "strategy_label", "n_months",
    "ann_mean", "ann_vol", "ann_sharpe", "ann_sortino",
    "mean_tstat", "hit_rate", "pnl_ratio", "cum_return",
    "alpha_m_vs_ew", "alpha_ann_vs_ew", "alpha_t_vs_ew",
    "beta_vs_ew", "beta_t_vs_ew", "corr_vs_ew",
]

for region_name in region_order:
    print(f"\n{region_name.upper()}")
    display(
        momentum_summary_all
        .loc[momentum_summary_all["region"] == region_name, summary_cols]
        .rename(columns={"strategy_label": "strategy"})
        .round(4)
    )

### Cumulative return plots

In [ ]:
def plot_cumulative_strategies(momentum_panels, region, strategy_list):
    df = momentum_panels[region].copy().sort_values("date")
    plt.figure(figsize=(12, 6))
    for strategy in strategy_list:
        if strategy not in df.columns:
            continue
        cum = (1.0 + df[strategy].fillna(0.0)).cumprod()
        ls = "--" if strategy == "ew_factor_ret" else "-"
        lw = 2.0 if strategy == "ew_factor_ret" else 1.5
        plt.plot(df["date"], cum, label=strategy_labels.get(strategy, strategy), linewidth=lw, linestyle=ls)
    plt.axhline(1.0, linestyle=":", linewidth=1)
    plt.title(f"{region.upper()} — cumulative returns")
    plt.xlabel("Date"); plt.ylabel("Growth of $1")
    plt.legend()
    plt.tight_layout()
    plt.show()


for region_name in region_order:
    plot_cumulative_strategies(momentum_panels_by_region, region_name, strategy_order)

## 7. Long-leg and short-leg decomposition

For each strategy and each month we compute the signal-weighted return of
the long leg (Winners) and the short leg (Losers) on a stand-alone basis, then
report the Winners − EW and EW − Losers spreads (Section 4.6.3).

In [ ]:
def _signal_weighted_return_of_leg(returns_df, weights_df, side="long"):
    if side == "long":
        leg_w = weights_df.where(weights_df > 0, 0.0)
    else:
        leg_w = (-weights_df).where(weights_df < 0, 0.0)
    leg_total = leg_w.sum(axis=1, skipna=True)
    weighted_ret = (leg_w * returns_df).sum(axis=1, skipna=True, min_count=1)
    return weighted_ret.where(leg_total > 0) / leg_total.where(leg_total > 0)


def build_winners_losers_panel(panel, weights_dict):
    fcols = get_factor_cols(panel)
    df = panel.sort_values("date").copy()
    returns_df = df[fcols]
    ew_ret = returns_df.mean(axis=1, skipna=True)

    out = df[["date"]].copy()
    out["ew_factor_ret"] = ew_ret.values

    for strat_name, w in weights_dict.items():
        w_aligned = w.reindex(index=returns_df.index, columns=returns_df.columns)
        winners_raw = _signal_weighted_return_of_leg(returns_df, w_aligned, side="long")
        losers_raw = _signal_weighted_return_of_leg(returns_df, w_aligned, side="short")

        out[f"{strat_name}_winners"] = winners_raw.values
        out[f"{strat_name}_losers"] = losers_raw.values
        out[f"{strat_name}_winners_xs_ew"] = (winners_raw - ew_ret).values
        out[f"{strat_name}_losers_xs_ew"] = (ew_ret - losers_raw).values

    return out


def performance_summary_with_skew(ret, periods_per_year=12):
    base = performance_summary(ret, periods_per_year=periods_per_year)
    s = pd.Series(ret).dropna().astype(float)
    base["skewness"] = float(_skew(s, bias=False)) if len(s) >= 3 else np.nan
    return base


def _sig_stars(t):
    if pd.isna(t):
        return ""
    a = abs(t)
    if a > 3.291: return "***"
    if a > 2.576: return "**"
    if a > 1.96:  return "*"
    if a > 1.66:  return " ."
    return ""


winners_losers_panels_by_region = {}
for region_name, df_region in region_series_map.items():
    weights = momentum_weights_by_region[region_name]
    wl_panel = build_winners_losers_panel(df_region, weights)

    # merge in combined strategy returns so all columns are available in one place
    strat_returns = momentum_panels_by_region[region_name][
        ["date", "tsmom_sign_1m", "tsmom_scaled_1m", "csmom_discrete_1m", "csmom_continuous_1m"]
    ]
    winners_losers_panels_by_region[region_name] = wl_panel.merge(strat_returns, on="date", how="left")

In [ ]:
strategy_blocks = [
    {"rows": [("ew_factor_ret", "Equal-weighted Factor Portfolio")]},
    {"rows": [
        ("tsmom_sign_1m",                "Time-series Factor Momentum — sign"),
        ("tsmom_sign_1m_winners",        "    Winners (raw)"),
        ("tsmom_sign_1m_losers",         "    Losers (raw)"),
        ("tsmom_sign_1m_winners_xs_ew",  "    Winners − EW"),
        ("tsmom_sign_1m_losers_xs_ew",   "    EW − Losers"),
    ]},
    {"rows": [
        ("tsmom_scaled_1m",                "Time-series Factor Momentum — scaled"),
        ("tsmom_scaled_1m_winners",        "    Winners (raw)"),
        ("tsmom_scaled_1m_losers",         "    Losers (raw)"),
        ("tsmom_scaled_1m_winners_xs_ew",  "    Winners − EW"),
        ("tsmom_scaled_1m_losers_xs_ew",   "    EW − Losers"),
    ]},
    {"rows": [
        ("csmom_discrete_1m",                "Cross-sectional Factor Momentum — discrete"),
        ("csmom_discrete_1m_winners",        "    Winners (raw)"),
        ("csmom_discrete_1m_losers",         "    Losers (raw)"),
        ("csmom_discrete_1m_winners_xs_ew",  "    Winners − EW"),
        ("csmom_discrete_1m_losers_xs_ew",   "    EW − Losers"),
    ]},
    {"rows": [
        ("csmom_continuous_1m",                "Cross-sectional Factor Momentum — continuous"),
        ("csmom_continuous_1m_winners",        "    Winners (raw)"),
        ("csmom_continuous_1m_losers",         "    Losers (raw)"),
        ("csmom_continuous_1m_winners_xs_ew",  "    Winners − EW"),
        ("csmom_continuous_1m_losers_xs_ew",   "    EW − Losers"),
    ]},
]


def build_winners_losers_table(panel_full):
    rows = []
    for block in strategy_blocks:
        for col, label in block["rows"]:
            if col not in panel_full.columns:
                continue
            perf = performance_summary_with_skew(panel_full[col])
            rows.append({
                "Strategy": label,
                "Return_ann_pct": f"{perf['ann_mean'] * 100:.2f}{_sig_stars(perf['mean_tstat'])}" if pd.notna(perf["ann_mean"]) else "",
                "Vol_ann_pct":    f"{perf['ann_vol'] * 100:.2f}" if pd.notna(perf["ann_vol"]) else "",
                "Sharpe":         f"{perf['ann_sharpe']:.2f}" if pd.notna(perf["ann_sharpe"]) else "",
                "Skewness":       f"{perf['skewness']:.2f}" if pd.notna(perf["skewness"]) else "",
                "t_value":        f"{perf['mean_tstat']:.2f}" if pd.notna(perf["mean_tstat"]) else "",
                "n_months":       perf["n_months"],
            })
    return pd.DataFrame(rows)


for region_name, full_panel in winners_losers_panels_by_region.items():
    print(f"\nWinners–Losers performance — {region_name.upper()}")
    display(build_winners_losers_table(full_panel))

## 8. TSMOM vs CSMOM spanning regressions

Two literature-consistent pairings:

- **Gupta–Kelly primary**: TSMOM scaled vs CSMOM continuous.
- **Ehsani–Linnainmaa robustness**: TSMOM sign vs CSMOM discrete.

For each pair, we run the HAC regression in both directions and report the
annualized alpha, beta, and $R^2$.

In [ ]:
def spanning_regression(y, x, maxlags=6):
    df = pd.DataFrame({"y": y, "x": x}).dropna()
    if len(df) < 24:
        return {k: np.nan for k in ["alpha_ann", "alpha_t", "beta", "beta_t", "r2"]} | {"n_obs": len(df)}

    X = sm.add_constant(df["x"], has_constant="add")
    fit = sm.OLS(df["y"], X).fit(cov_type="HAC", cov_kwds={"maxlags": maxlags})
    return {
        "n_obs": int(fit.nobs),
        "alpha_m": float(fit.params["const"]),
        "alpha_ann": float(fit.params["const"] * 12.0),
        "alpha_t": float(fit.tvalues["const"]),
        "beta": float(fit.params["x"]),
        "beta_t": float(fit.tvalues["x"]),
        "r2": float(fit.rsquared),
    }


ts_cs_pairs = {
    "GK_primary_scaled_vs_continuous": ("tsmom_scaled_1m", "csmom_continuous_1m"),
    "EL_robustness_sign_vs_discrete":  ("tsmom_sign_1m",   "csmom_discrete_1m"),
}

ts_cs_rows = []
for region_name, panel in momentum_panels_by_region.items():
    for pair_label, (ts_col, cs_col) in ts_cs_pairs.items():
        for dep, rhs in [(ts_col, cs_col), (cs_col, ts_col)]:
            reg = spanning_regression(panel[dep], panel[rhs], maxlags=6)
            corr = panel[[dep, rhs]].corr().iloc[0, 1]
            ts_cs_rows.append({
                "region": region_name, "pair": pair_label,
                "dependent": dep, "rhs": rhs,
                "corr": corr,
                "alpha_ann": reg["alpha_ann"], "alpha_t": reg["alpha_t"],
                "beta": reg["beta"], "beta_t": reg["beta_t"],
                "r2": reg["r2"], "n_obs": reg["n_obs"],
            })

ts_cs_spanning = pd.DataFrame(ts_cs_rows).round(4)
for pair_label in ts_cs_pairs.keys():
    print(f"\n{pair_label}")
    display(ts_cs_spanning.loc[ts_cs_spanning["pair"] == pair_label].sort_values(["region", "dependent"]).reset_index(drop=True))

## 9. Decomposition of momentum profits

We decompose the raw (unnormalized) returns of the continuous TS and CS
strategies into autocovariance, cross-factor covariance, and mean components
using the Moskowitz–Lo–MacKinlay–Lewellen framework (Section 4.7):

$$
\mathbb{E}[\pi^{TS}_t] = \frac{\operatorname{tr}(\Omega)}{N} + \frac{m'm}{N}, \qquad
\mathbb{E}[\pi^{CS}_t] = \frac{\operatorname{tr}(\Omega)}{N} - \frac{1}{N^2}\sum_{i\ne j} \omega_{ij} + \sigma_m^2.
$$

$\Omega$ has entries $\omega_{ij} = \operatorname{Cov}(r_{i,t-1}, r_{j,t})$ and
$m_i$ is the unconditional mean of factor $i$. Components are annualized by
multiplying monthly estimates by 12.

In [ ]:
def pairwise_cov_lag_current(df_ret, factor_cols):
    R = df_ret.sort_values("date")[factor_cols].copy().astype(float)
    X = R.shift(1)
    Y = R.copy()
    n = len(factor_cols)
    Omega = np.full((n, n), np.nan)

    for i, fi in enumerate(factor_cols):
        for j, fj in enumerate(factor_cols):
            sub = pd.concat([X[fi], Y[fj]], axis=1).dropna()
            if len(sub) >= 12:
                Omega[i, j] = np.cov(sub.iloc[:, 0], sub.iloc[:, 1], ddof=1)[0, 1]

    return Omega


def compute_realized_returns(panel, factor_cols):
    R = panel.sort_values("date")[factor_cols].copy().astype(float)
    R_lag = R.shift(1)
    N = len(factor_cols)
    ts_monthly = (R_lag * R).sum(axis=1) / N
    R_lag_demeaned = R_lag.sub(R_lag.mean(axis=1), axis=0)
    cs_monthly = (R_lag_demeaned * R).sum(axis=1) / N
    return ts_monthly.dropna().mean() * 12, cs_monthly.dropna().mean() * 12


def decompose_factor_momentum_region(panel):
    fcols = get_factor_cols(panel)
    R = panel.sort_values("date")[fcols].copy().astype(float)
    m = R.mean(axis=0, skipna=True).to_numpy(dtype=float)
    N = len(fcols)

    Omega = pairwise_cov_lag_current(panel, fcols)
    Omega_filled = np.where(np.isfinite(Omega), Omega, 0.0)

    trace_omega = np.trace(Omega_filled)
    offdiag_sum = Omega_filled.sum() - trace_omega

    auto = trace_omega / N
    cross = -offdiag_sum / (N**2)
    mean_disp = np.var(m, ddof=0)
    mean_sq = np.mean(m**2)

    ts_total = auto + mean_sq
    cs_total = auto + cross + mean_disp
    ts_realized_ann, cs_realized_ann = compute_realized_returns(panel, fcols)

    return pd.DataFrame([
        {
            "strategy": "TS_continuous_raw",
            "auto_ann": 12 * auto,
            "cross_ann": np.nan,
            "mean_component_ann": 12 * mean_sq,
            "total_ann": 12 * ts_total,
            "realized_ann": ts_realized_ann,
            "auto_share": auto / ts_total if ts_total else np.nan,
            "cross_share": np.nan,
            "mean_share": mean_sq / ts_total if ts_total else np.nan,
        },
        {
            "strategy": "CS_continuous_raw",
            "auto_ann": 12 * auto,
            "cross_ann": 12 * cross,
            "mean_component_ann": 12 * mean_disp,
            "total_ann": 12 * cs_total,
            "realized_ann": cs_realized_ann,
            "auto_share": auto / cs_total if cs_total else np.nan,
            "cross_share": cross / cs_total if cs_total else np.nan,
            "mean_share": mean_disp / cs_total if cs_total else np.nan,
        },
    ])


decomp_tables = []
for region_name, panel in region_series_map.items():
    dec = decompose_factor_momentum_region(panel)
    dec.insert(0, "region", region_name)
    decomp_tables.append(dec)

decomp_all = pd.concat(decomp_tables, axis=0, ignore_index=True).round(6)
display(decomp_all)

## 10. Rolling and conditional-state analysis

A 36-month rolling window is used to compute, in each region, the
cross-sectional average lag-1 autocorrelation across factors. Each month is
classified as a high- or low-autocorrelation state via a median split, and
performance is reported separately within each state for the two continuous
strategies (TSMOM scaled and CSMOM continuous), which are the strategies for
which Section 9's decomposition formally connects profits to autocovariance.

In [ ]:
ROLL_WIN = 36
TS_PRIMARY = "tsmom_scaled_1m"
CS_PRIMARY = "csmom_continuous_1m"


def rolling_sharpe(series, window=36, periods_per_year=12):
    s = pd.Series(series).astype(float)
    roll_mean = s.rolling(window).mean() * periods_per_year
    roll_vol = s.rolling(window).std(ddof=1) * np.sqrt(periods_per_year)
    return roll_mean / roll_vol


def rolling_max_drawdown(series, window=36):
    s = pd.Series(series).fillna(0.0).astype(float)
    vals = []
    for end in range(len(s)):
        if end + 1 < window:
            vals.append(np.nan)
            continue
        sub = s.iloc[end - window + 1:end + 1]
        wealth = (1.0 + sub).cumprod()
        dd = wealth / wealth.cummax() - 1.0
        vals.append(dd.min())
    return pd.Series(vals, index=s.index)


def rolling_mean_ar1(panel, window=36):
    df = panel.sort_values("date").copy()
    fcols = get_factor_cols(df)
    values = []
    for end in range(len(df)):
        if end + 1 < window:
            values.append(np.nan)
            continue
        sub = df.iloc[end - window + 1:end + 1]
        ar1_list = []
        for f in fcols:
            s = pd.Series(sub[f]).dropna().astype(float)
            if len(s) >= 12:
                tmp = pd.concat([s, s.shift(1)], axis=1).dropna()
                if len(tmp) >= 12:
                    ar1_list.append(tmp.iloc[:, 0].corr(tmp.iloc[:, 1]))
        values.append(np.nanmean(ar1_list) if len(ar1_list) > 0 else np.nan)
    return pd.Series(values, index=df.index)


def rolling_alpha_vs_benchmark(y, x, window=36, maxlags=6):
    df = pd.DataFrame({"y": y, "x": x})
    out = []
    for end in range(len(df)):
        sub = df.iloc[max(0, end - window + 1):end + 1].dropna()
        if len(sub) < max(24, window // 2):
            out.append(np.nan)
            continue
        X = sm.add_constant(sub["x"], has_constant="add")
        fit = sm.OLS(sub["y"], X).fit(cov_type="HAC", cov_kwds={"maxlags": maxlags})
        out.append(float(fit.params["const"] * 12.0))
    return pd.Series(out, index=df.index)

In [ ]:
rolling_outputs_by_region = {}
state_tables = []

for region_name, strat_panel in momentum_panels_by_region.items():
    panel = strat_panel.sort_values("date").copy()
    panel["roll_sharpe_ts"] = rolling_sharpe(panel[TS_PRIMARY], window=ROLL_WIN)
    panel["roll_sharpe_cs"] = rolling_sharpe(panel[CS_PRIMARY], window=ROLL_WIN)
    panel["roll_mean_ar1"] = rolling_mean_ar1(region_series_map[region_name], window=ROLL_WIN)
    panel["roll_alpha_ts_vs_ew"] = rolling_alpha_vs_benchmark(panel[TS_PRIMARY], panel["ew_factor_ret"], window=ROLL_WIN, maxlags=6)
    panel["roll_dd_ts"] = rolling_max_drawdown(panel[TS_PRIMARY], window=ROLL_WIN)

    med_ar1 = panel["roll_mean_ar1"].median(skipna=True)
    panel["high_autocorr"] = np.where(panel["roll_mean_ar1"] >= med_ar1, 1, 0)
    panel.loc[pd.isna(panel["roll_mean_ar1"]), "high_autocorr"] = np.nan

    for label, mask in {"High autocorr": panel["high_autocorr"] == 1,
                        "Low autocorr": panel["high_autocorr"] == 0}.items():
        for strat in [TS_PRIMARY, CS_PRIMARY]:
            perf = performance_summary(panel.loc[mask, strat])
            state_tables.append({
                "region": region_name, "state": label, "strategy": strat,
                "n_months": perf["n_months"],
                "ann_mean": perf["ann_mean"], "ann_vol": perf["ann_vol"],
                "ann_sharpe": perf["ann_sharpe"], "ann_sortino": perf["ann_sortino"],
                "hit_rate": perf["hit_rate"],
            })

    rolling_outputs_by_region[region_name] = panel

rolling_state_summary = pd.DataFrame(state_tables).round(4)
display(rolling_state_summary)

In [ ]:
for region_name, panel in rolling_outputs_by_region.items():
    fig, axes = plt.subplots(3, 1, figsize=(12, 9), sharex=True)

    axes[0].plot(panel["date"], panel["roll_sharpe_ts"], label="TSMOM scaled", linewidth=1.5)
    axes[0].plot(panel["date"], panel["roll_sharpe_cs"], label="CSMOM continuous", linewidth=1.5)
    axes[0].axhline(0, linestyle=":", linewidth=1)
    axes[0].set_title(f"{region_name.upper()} — Rolling {ROLL_WIN}m Sharpe")
    axes[0].legend()

    axes[1].plot(panel["date"], panel["roll_mean_ar1"], label="Mean AR(1) across factors", linewidth=1.5)
    axes[1].axhline(panel["roll_mean_ar1"].median(skipna=True), linestyle="--", linewidth=1, label="Median state split")
    axes[1].set_title(f"{region_name.upper()} — Rolling mean AR(1)")
    axes[1].legend()

    axes[2].plot(panel["date"], panel["roll_alpha_ts_vs_ew"], label="TSMOM alpha vs EW (ann.)", linewidth=1.5)
    axes[2].plot(panel["date"], panel["roll_dd_ts"], label="TSMOM rolling max drawdown", linewidth=1.2)
    axes[2].axhline(0, linestyle=":", linewidth=1)
    axes[2].set_title(f"{region_name.upper()} — Rolling alpha and drawdown")
    axes[2].legend()

    plt.tight_layout()
    plt.show()

## 11. Decade subsample stability

The pipeline is re-run on non-overlapping decade-style blocks:
1986–1989, 1990s, 2000s, 2010s, and 2020–2024 (Section 4.8.2). Strategies are
rebuilt from scratch in each subsample. The two figures below show the
annualized Sharpe ratio and the annualized alpha against the EW benchmark for
each decade, region, and active strategy.

In [ ]:
def get_decade_subperiods(df_region):
    df = df_region.sort_values("date").copy()
    years = pd.Series(df["date"]).dropna().dt.year
    if years.empty:
        return OrderedDict()

    min_year, max_year = int(years.min()), int(years.max())
    candidates = [
        (1986, 1989, "1986-1989"),
        (1990, 1999, "1990s"),
        (2000, 2009, "2000s"),
        (2010, 2019, "2010s"),
        (2020, 2029, "2020s"),
    ]
    out = OrderedDict()
    for start_y, end_y, label in candidates:
        if end_y < min_year or start_y > max_year:
            continue
        start_eff = max(start_y, min_year)
        end_eff = min(end_y, max_year)
        sub = df[(df["date"].dt.year >= start_eff) & (df["date"].dt.year <= end_eff)].copy()
        if len(sub) == 0:
            continue
        if label == "2020s" and end_eff < 2029:
            final_label = f"{start_eff}-{end_eff}"
        elif label == "1986-1989":
            final_label = f"{start_eff}-{end_eff}"
        else:
            final_label = label
        out[final_label] = sub
    return out


decade_summaries = []
for region_name, df_region in region_series_map.items():
    subperiods = get_decade_subperiods(df_region)
    for sample_label, df_sub in subperiods.items():
        strat_panel, _ = build_factor_momentum_panel(df_sub)
        summary_df = summarize_factor_momentum_panel(strat_panel, region_name)
        summary_df["sample_label"] = sample_label
        decade_summaries.append(summary_df)

decade_summary_long = pd.concat(decade_summaries, axis=0, ignore_index=True)
decade_summary_long["strategy"] = pd.Categorical(decade_summary_long["strategy"], categories=strategy_order, ordered=True)
decade_summary_long = decade_summary_long.sort_values(["region", "sample_label", "strategy"]).reset_index(drop=True)
decade_summary_long["strategy_label"] = decade_summary_long["strategy"].map(strategy_labels)

display(decade_summary_long[["region", "sample_label", "strategy_label", "ann_mean", "ann_vol", "ann_sharpe", "alpha_ann_vs_ew", "alpha_t_vs_ew"]].round(4))

In [ ]:
decade_order = ["1986-1989", "1990s", "2000s", "2010s", "2020-2024"]

strategy_alias = {
    "TSMOM sign (1-1)": "TS sign",
    "TSMOM scaled (1-1)": "TS scaled",
    "CSMOM discrete (1-1)": "CS discrete",
    "CSMOM continuous (1-1)": "CS continuous",
}
cols = ["TS sign", "TS scaled", "CS discrete", "CS continuous"]
colors = {
    "TS sign": "#0B1F3A",
    "TS scaled": "#244A7A",
    "CS discrete": "#3A3A3A",
    "CS continuous": "#8A8A8A",
}


def plot_decade_bars(metric, ylabel, title):
    plot_df = decade_summary_long.copy()
    plot_df["strategy_short"] = plot_df["strategy_label"].map(strategy_alias)
    plot_df = plot_df[plot_df["strategy_short"].isin(cols)].copy()
    plot_df["sample_label"] = pd.Categorical(plot_df["sample_label"], categories=decade_order, ordered=True)
    plot_df = plot_df.groupby(["region", "sample_label", "strategy_short"], as_index=False)[metric].mean()

    fig, axes = plt.subplots(2, 2, figsize=(17, 10), sharey=True)
    axes = axes.flatten()
    bar_w = 0.18
    offsets = np.linspace(-1.5 * bar_w, 1.5 * bar_w, len(cols))

    for ax, region_name in zip(axes, region_order):
        sub = plot_df[plot_df["region"] == region_name].copy()
        pivot = (
            sub.pivot(index="sample_label", columns="strategy_short", values=metric)
            .reindex(decade_order).reindex(columns=cols)
        )
        pivot = pivot.loc[pivot.notna().any(axis=1)]
        x = np.arange(len(pivot.index))

        for c, off in zip(cols, offsets):
            y = pivot[c].values
            mask = ~pd.isna(y)
            ax.bar(x[mask] + off, y[mask], width=bar_w * 0.92,
                   label=c, color=colors[c], edgecolor="black", linewidth=0.55)

        ax.set_xticks(x)
        ax.set_xticklabels(pivot.index, fontsize=10)
        ax.set_title(region_name.upper(), fontsize=12, pad=8)
        ax.axhline(0, color="black", linewidth=1)
        ax.spines["top"].set_visible(False)
        ax.spines["right"].set_visible(False)

    axes[0].set_ylabel(ylabel)
    axes[2].set_ylabel(ylabel)

    handles, labels = axes[0].get_legend_handles_labels()
    by_label = dict(zip(labels, handles))
    fig.legend(by_label.values(), by_label.keys(),
               loc="upper center", ncol=4, frameon=False, bbox_to_anchor=(0.5, 0.98))
    fig.suptitle(title, fontsize=16, y=0.995)
    plt.tight_layout(rect=[0, 0, 1, 0.94])
    plt.show()


plot_decade_bars("ann_sharpe", "Annualized Sharpe ratio", "Sharpe Ratios Through Time by Region")
plot_decade_bars("alpha_ann_vs_ew", "Annualized alpha vs EW", "Annualized Alpha vs EW Through Time by Region")

## 12. Transaction costs and net performance

Monthly turnover is computed at the factor-allocation layer using drifted
weights $\tilde w_{i,t-1} = w_{i,t-1}(1 + r_{i,t-1}) / \sum_j w_{j,t-1}(1 + r_{j,t-1})$
and target weights $w_{i,t}$. The net return is
$r^{\text{net}}_t = r^{\text{gross}}_t - c \cdot \text{TO}_t$ with
$c \in \{10, 25, 50\}$ bps. The same procedure is applied to the equal-weight
benchmark so that the comparison is net-vs-net.

In [ ]:
ALPHA_COST_SCENARIOS_BPS = [10, 25, 50]

ALPHA_STRATEGIES = [
    "tsmom_sign_1m", "tsmom_scaled_1m",
    "csmom_discrete_1m", "csmom_continuous_1m",
]


def get_raw_factor_cols(raw_returns_panel):
    exclude = {"date", "location"}
    return [c for c in raw_returns_panel.columns
            if c not in exclude and pd.api.types.is_numeric_dtype(raw_returns_panel[c])]


def align_weights_and_raw_returns(weights_df, raw_returns_panel):
    raw_panel = raw_returns_panel.sort_values("date").copy()
    common = [c for c in weights_df.columns if c in get_raw_factor_cols(raw_panel)]
    w = weights_df[common].copy()
    r = raw_panel[["date"] + common].copy()
    w.index = r.index
    return r, common, w


def post_return_weights(w_t, r_t):
    gross_position = w_t * (1.0 + r_t)
    gross_abs = np.abs(gross_position).sum()
    if (not np.isfinite(gross_abs)) or gross_abs <= 0:
        return pd.Series(np.nan, index=w_t.index)
    return gross_position / gross_abs


def compute_turnover_series(weights_df, raw_returns_panel):
    r_panel, fcols, w = align_weights_and_raw_returns(weights_df, raw_returns_panel)
    turnover = pd.Series(np.nan, index=r_panel.index, dtype=float)
    prev_post = None

    for t in range(len(r_panel)):
        w_t = w.iloc[t]
        r_t = r_panel.loc[r_panel.index[t], fcols].astype(float)
        if prev_post is not None:
            valid = pd.concat([w_t, prev_post], axis=1).dropna()
            if len(valid) > 0:
                turnover.iloc[t] = np.abs(valid.iloc[:, 0] - valid.iloc[:, 1]).sum()
        prev_post = post_return_weights(w_t, r_t) if w_t.notna().sum() > 0 else None

    return pd.DataFrame({"date": r_panel["date"].values, "turnover": turnover.values})


def apply_transaction_costs(gross_ret, turnover, cost_bps):
    cost = turnover * (cost_bps / 10000.0)
    return gross_ret - cost, cost


def build_equal_weight_overlay_weights(raw_returns_panel):
    df = raw_returns_panel.sort_values("date").copy()
    fcols = get_raw_factor_cols(df)
    weights = pd.DataFrame(index=df.index, columns=fcols, dtype=float)
    for i in df.index:
        avail = df.loc[i, fcols].notna()
        n_avail = int(avail.sum())
        if n_avail > 0:
            weights.loc[i, avail] = 1.0 / n_avail
            weights.loc[i, ~avail] = np.nan
        else:
            weights.loc[i, :] = np.nan
    return weights


def hac_alpha_regression(y, x, maxlags=6):
    df = pd.DataFrame({"y": y, "x": x}).dropna()
    if len(df) < 24:
        return {"n_obs": len(df), "alpha_ann": np.nan, "alpha_t": np.nan,
                "beta": np.nan, "beta_t": np.nan, "r2": np.nan}
    X = sm.add_constant(df["x"], has_constant="add")
    fit = sm.OLS(df["y"], X).fit(cov_type="HAC", cov_kwds={"maxlags": maxlags})
    return {
        "n_obs": int(fit.nobs),
        "alpha_ann": float(fit.params["const"] * 12.0),
        "alpha_t": float(fit.tvalues["const"]),
        "beta": float(fit.params["x"]),
        "beta_t": float(fit.tvalues["x"]),
        "r2": float(fit.rsquared),
    }

In [ ]:
alpha_cost_rows = []

for region_name in region_series_map.keys():
    strategy_panel = momentum_panels_by_region[region_name].sort_values("date").copy()
    raw_returns_panel = region_series_map[region_name].sort_values("date").copy()
    region_weights = momentum_weights_by_region[region_name]

    ew_weights = build_equal_weight_overlay_weights(raw_returns_panel)
    ew_turnover = compute_turnover_series(ew_weights, raw_returns_panel).rename(columns={"turnover": "ew_turnover"})

    for cost_bps in ALPHA_COST_SCENARIOS_BPS:
        ew_df = strategy_panel[["date", "ew_factor_ret"]].merge(ew_turnover, on="date", how="left")
        ew_net, _ = apply_transaction_costs(ew_df["ew_factor_ret"], ew_df["ew_turnover"], cost_bps)
        ew_df["ew_net"] = ew_net

        for strat in ALPHA_STRATEGIES:
            strat_turnover = compute_turnover_series(region_weights[strat], raw_returns_panel).rename(columns={"turnover": "strat_turnover"})
            strat_df = strategy_panel[["date", strat, "ew_factor_ret"]].merge(strat_turnover, on="date", how="left")
            strategy_net, _ = apply_transaction_costs(strat_df[strat], strat_df["strat_turnover"], cost_bps)
            strat_df["strategy_net"] = strategy_net

            merged = strat_df.merge(ew_df[["date", "ew_net"]], on="date", how="left")

            gross_vs_gross = hac_alpha_regression(merged[strat], merged["ew_factor_ret"], maxlags=6)
            net_vs_gross   = hac_alpha_regression(merged["strategy_net"], merged["ew_factor_ret"], maxlags=6)
            net_vs_net     = hac_alpha_regression(merged["strategy_net"], merged["ew_net"], maxlags=6)

            alpha_cost_rows.append({
                "region": region_name, "strategy": strat, "cost_bps": cost_bps,
                "strategy_ann_turnover": merged["strat_turnover"].mean(skipna=True) * 12.0,
                "ew_ann_turnover": ew_df["ew_turnover"].mean(skipna=True) * 12.0,
                "gross_alpha_vs_grossEW_ann": gross_vs_gross["alpha_ann"],
                "gross_alpha_vs_grossEW_t":   gross_vs_gross["alpha_t"],
                "net_alpha_vs_grossEW_ann":   net_vs_gross["alpha_ann"],
                "net_alpha_vs_grossEW_t":     net_vs_gross["alpha_t"],
                "net_alpha_vs_netEW_ann":     net_vs_net["alpha_ann"],
                "net_alpha_vs_netEW_t":       net_vs_net["alpha_t"],
                "n_obs": net_vs_net["n_obs"],
            })

alpha_after_costs_all = pd.DataFrame(alpha_cost_rows).round(4)

In [ ]:
print("ALPHA AFTER COSTS — 10 bps")
display(
    alpha_after_costs_all
    .loc[alpha_after_costs_all["cost_bps"] == 10]
    .sort_values(["region", "strategy"])
    .reset_index(drop=True)
)

print("\nALPHA AFTER COSTS — ALL COST SCENARIOS")
display(
    alpha_after_costs_all
    .sort_values(["cost_bps", "region", "strategy"])
    .reset_index(drop=True)
)